In [1]:
import oceanbench

oceanbench.__version__

'0.0.3'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# SPDX-FileCopyrightText: 2025 Mercator Ocean International <https://www.mercator-ocean.eu/>
#
# SPDX-License-Identifier: EUPL-1.2

# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray

challenger_dataset: xarray.Dataset = (
    xarray.open_mfdataset(
        [
            "https://minio.dive.edito.eu/project-moi-glo36-oceanbench/public/20230104.zarr",
        ],
        engine="zarr",
        combine="nested",
        concat_dim="first_day_datetime",
        parallel=True,
    )
    .rename({"lat": "latitude", "lon": "longitude"})
    .unify_chunks()
    .assign({"first_day_datetime": [datetime.fromisoformat("2023-01-04")]})
)


challenger_dataset["zos"].attrs["standard_name"] = "sea_surface_height_above_geoid"
challenger_dataset["thetao"].attrs["standard_name"] = "sea_water_potential_temperature"
challenger_dataset["so"].attrs["standard_name"] = "sea_water_salinity"
challenger_dataset["uo"].attrs["standard_name"] = "eastward_sea_water_velocity"
challenger_dataset["vo"].attrs["standard_name"] = "northward_sea_water_velocity"
challenger_dataset["latitude"].attrs["standard_name"] = "latitude"
challenger_dataset["longitude"].attrs["standard_name"] = "longitude"

challenger_dataset

print(challenger_dataset)


<xarray.Dataset> Size: 50GB
Dimensions:             (first_day_datetime: 1, lead_day_index: 7, depth: 50,
                         latitude: 2041, longitude: 4320)
Coordinates:
  * depth               (depth) float32 200B 0.494 1.541 ... 5.275e+03 5.728e+03
  * latitude            (latitude) float32 8kB -80.0 -79.92 ... 89.92 90.0
  * lead_day_index      (lead_day_index) int64 56B 0 1 2 3 4 5 6
  * longitude           (longitude) float32 17kB -180.0 -179.9 ... 179.8 179.9
  * first_day_datetime  (first_day_datetime) datetime64[us] 8B 2023-01-04
Data variables:
    so                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.array<chunksize=(1, 1, 7, 256, 540), meta=np.ndarray>
    thetao              (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.array<chunksize=(1, 1, 7, 256, 540), meta=np.ndarray>
    uo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.arra

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [3]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Surface height (m) [sea_surface_height_above_geoid]{surface},0.185129,0.188180,0.191729,0.196628,0.187584,0.179020,0.177233
Surface temperature (°C) [sea_water_potential_temperature]{surface},0.754099,0.746241,0.741034,0.734648,0.733596,0.734350,0.732862
Surface salinity (PSU) [sea_water_salinity]{surface},0.743157,0.747366,0.749267,0.744462,0.744975,0.748035,0.749087
Surface northward velocity (m/s) [northward_sea_water_velocity]{surface},0.152241,0.154100,0.156448,0.158948,0.161531,0.164117,0.167407
Surface eastward velocity (m/s) [eastward_sea_water_velocity]{surface},0.151815,0.154183,0.156614,0.158860,0.161653,0.164104,0.167584
50m temperature (°C) [sea_water_potential_temperature]{50m},0.901110,0.904672,0.904939,0.906343,0.914996,0.920984,0.928509
50m salinity (PSU) [sea_water_salinity]{50m},0.412407,0.412433,0.412295,0.412117,0.412261,0.412350,0.412401
50m northward velocity (m/s) [northward_sea_water_velocity]{50m},0.149004,0.150827,0.152387,0.154077,0.155421,0.156994,0.158633
50m eastward velocity (m/s) [eastward_sea_water_velocity]{50m},0.147528,0.149717,0.151387,0.152616,0.154039,0.155645,0.157695
100m temperature (°C) [sea_water_potential_temperature]{100m},0.984619,0.986458,0.990043,0.991361,0.996498,1.002191,1.008225


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},23.881464,23.562943,23.736929,23.977169,23.805344,24.049902,23.761709


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Northward geostrophic velocity (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.158405,0.159994,0.162545,0.165205,0.166449,0.167696,0.167974
Eastward geostrophic velocity (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.160622,0.163275,0.163623,0.164440,0.165561,0.167537,0.170380


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(challenger_dataset)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6
Surface Lagrangian trajectory deviation (km) [],13.253161,25.582176,36.984825,47.554512,57.390961


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Surface height (m) [sea_surface_height_above_geoid]{surface},0.151541,0.153614,0.155053,0.153516,0.149113,0.143732,0.143426
Surface temperature (°C) [sea_water_potential_temperature]{surface},0.453199,0.462800,0.473441,0.484666,0.499040,0.520511,0.542959
Surface salinity (PSU) [sea_water_salinity]{surface},0.388226,0.395949,0.402756,0.406164,0.412359,0.422003,0.430482
Surface northward velocity (m/s) [northward_sea_water_velocity]{surface},0.132784,0.135970,0.140237,0.144683,0.149794,0.154620,0.161283
Surface eastward velocity (m/s) [eastward_sea_water_velocity]{surface},0.130265,0.134574,0.139228,0.143933,0.149694,0.154438,0.160230
50m temperature (°C) [sea_water_potential_temperature]{50m},0.609839,0.627200,0.639050,0.652733,0.671973,0.691808,0.715111
50m salinity (PSU) [sea_water_salinity]{50m},0.277457,0.279016,0.280666,0.282221,0.283977,0.285990,0.288367
50m northward velocity (m/s) [northward_sea_water_velocity]{50m},0.128679,0.131784,0.135225,0.139063,0.142819,0.146437,0.150265
50m eastward velocity (m/s) [eastward_sea_water_velocity]{50m},0.125880,0.129889,0.133296,0.136822,0.140829,0.144629,0.148826
100m temperature (°C) [sea_water_potential_temperature]{100m},0.642933,0.658132,0.674955,0.695246,0.718772,0.744592,0.770218


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [8]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},20.748575,20.847832,21.352081,22.072031,22.928757,23.962143,24.669508


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(challenger_dataset)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Northward geostrophic velocity (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.152344,0.155222,0.159149,0.163620,0.167211,0.168653,0.169900
Eastward geostrophic velocity (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.156091,0.160389,0.161476,0.164143,0.168046,0.171323,0.173638


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [10]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(challenger_dataset)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6
Surface Lagrangian trajectory deviation (km) [],10.29295,20.056608,29.213963,37.965965,46.350864


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [11]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(challenger_dataset)

,Message
0,Observation-based Class IV scores were not com...
